# 05 — Servir el modelo fine-tuneado

**Level 4 — Model Ops & Optimization**

Comparamos el modelo **BASE** contra el **fine-tuneado** (base + adaptador
LoRA de la section 3) con las mismas preguntas. PEFT inyecta el adaptador
EN el modelo recibido: por eso usamos **dos instancias separadas**.

In [1]:
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"
RUTA_ADAPTER = Path("../adapters/lora")

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cargar tokenizer + dos instancias

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

base = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float32)
base.to(dispositivo).eval()

ft_copia = AutoModelForCausalLM.from_pretrained(MODELO_BASE, torch_dtype=torch.float32)
ft_copia.to(dispositivo)
ft = PeftModel.from_pretrained(ft_copia, str(RUTA_ADAPTER))
ft.eval()
print("Base y FT cargados (instancias separadas)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  76%|███████▌  | 219/290 [00:00<00:00, 2184.08it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2089.27it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2936.62it/s]

Base y FT cargados (instancias separadas)


## La función de respuesta

In [3]:
def responder(modelo, pregunta: str, max_tokens: int = 80) -> str:
    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres 'librarian', un asistente tecnico que responde "
                "en espanol de forma breve y clara."
            ),
        },
        {"role": "user", "content": pregunta},
    ]
    prompt = tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(
            entrada["input_ids"], max_new_tokens=max_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        salida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

## Comparativa: BASE vs FT

In [4]:
preguntas = [
    "Que es una API REST?",
    "Como funciona el RAG?",
]

for pregunta in preguntas:
    print(f"\nPregunta: {pregunta}")
    print(f"BASE> {responder(base, pregunta)[:180]}")
    print(f"FT  > {responder(ft, pregunta)[:180]}")


Pregunta: Que es una API REST?


BASE> Un API REST es una abstracción de la realidad real de un objeto en línea y la creación de una interfaz de entorno para la realización de servicios y servicios de entorno.

En españ


FT  > Este es un modelo de servicio que utiliza REST para realizar servicios web.

Pregunta: Como funciona el RAG?


BASE> El RAG, el Régimen de Accesibilidad de la Agua, es un proceso de accesibilidad que se realiza durante el proceso de desarrollo de un proyecto de agua. La accesibilidad es la capaci


FT  > Eres 'librarian', un asistente tecnico que responde en espanol de forma breve y clara.

El RAG es un programa de trabajo que sirve para trabajar con información en formato de texto


## Conclusión

- **BASE** alucina (sin conocimiento del dominio)
- **FT** adoptó la persona librarian y su estilo (aprendió formato y tono)
- Con 10 ejemplos el FT NO revoluciona la calidad factual — eso lo mide el benchmark de la section 7